# RAG 01 — 建立向量資料庫（交通規則罰則問答・案 A）

- 負責人：Muse（issue #3）；基底：老師模板 yenlung.me/rag01colab（yenlung/Generative-AI《RAG_01_打造你的_RAG_系統.ipynb》）
- 版控規則見 colab/README.md：主檔 commit 前清 output；執行版由 PM 存 colab/executed/
- 本格不呼叫 LLM、不讀取任何密鑰；模型下載若要求授權，請以 Colab Secrets 設定 HF_TOKEN 後重跑模型格（勿將 token 寫入 notebook）

## 改編清單（老師模板 → 本案，SDD v1.0 定版）

- 教材 16 篇教學摘要 → 法規語料：core 小樣本 5 份起步，pending 3 份隔離；驗收通過後擴全量
- 單庫 → 雙庫物理隔離：core 主庫與 pending 隔離庫各自獨立 FAISS 索引與 zip；B 模式不載入 pending
- doc_id 流水號 → chunk_id（core:/pending: 前綴）作 doc_id，引用與 chunk 一一對應（SDD：生成引用須與命中 chunk_id 一致，模型自編條號即失敗）
- 匯出檔名 vector_embedding.zip → rag-data.zip（core）＋ rag-data-pending.zip（作業七規定）
- metadata 由 txt 標頭欄位＋manifest join：manifest 為狀態與 chunk_id 的權威，txt 為語料文字的權威；兩者不一致即 fail-closed
- source_ref：manifest 目前無此欄（見 tests/rag01_acceptance_report.md FINDING-01），向量 metadata 保留 source_url 並標記 source_ref 缺失，不捏造

## 流程總覽

1. 上傳 core 語料 zip ＋ core_manifest.json；pending 隔離包 ＋ pending_manifest.json 另行上傳（兩包路徑分開，絕不混入）
2. 解壓、載入語料檔；每檔以 chunk_id join manifest（無 manifest 條目者 fail-closed 拒絕；manifest 有條目但未上傳者警告跳過——小樣本模式，manifest 為全量覆蓋表）
3. Embedding：google/embeddinggemma-2（課程指定，勿換）；DOCUMENT_TEMPLATE 保留老師版（title: none，因 text 已含標題）
4. 建向量庫：core 主庫；pending 另建隔離庫（物理隔離，不共用索引）
5. 依作業七規定匯出 rag-data.zip（core）＋ rag-data-pending.zip（各含 index.faiss ＋ documents.json ＋ index_doc_ids.json ＋ config.json，老師版四檔格式）
6. 兩個 zip 上傳雲端轉可下載網址，網址填入 RAG 02 的下載 cell（repo 為 private，勿用 GitHub raw link）
7. 驗收：重載驗證（雜湊、數量、ID 對照、保存前後檢索一致）＋試查（T1 數字題取回、T4 連動、pending 隔離；T2 查無題不得幻覺由 RAG 02 生成層負責，此處僅驗檢索層——相似度不是正確率，老師版提醒）

## 1. 安裝套件

固定套件沿用老師版，勿自行增減；未引入固定套件總分 -1。安裝後若本執行階段已載入其他版本，請重啟執行階段再繼續（老師版提醒）。

In [ ]:
%pip install -q "sentence-transformers==6.1.0" "transformers==5.20.1" "faiss-cpu==1.15.1"

## 2. 檢查環境與集中設定

- `DEVICE`／`DTYPE` 沿用老師版：T4 不支援原生 bfloat16 時改用 float32；不使用 EmbeddingGemma 2 不相容的 float16。
- `TOKEN_LIMIT = 2048`：embeddinggemma-2 輸入上限；小樣本實測最長 1061 tokens（PR #6），未超限。

In [ ]:
import hashlib
import importlib.metadata as package_metadata
import io
import json
import re
import tempfile
import zipfile
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import torch
import faiss
from IPython.display import display
from sentence_transformers import SentenceTransformer
from huggingface_hub import HfApi

MODEL_ID = "google/embeddinggemma-2"
BATCH_SIZE = 4
TOKEN_LIMIT = 2048

# text 本身已含標題；這裡不再從 metadata 添加標題（老師版原註）。
DOCUMENT_TEMPLATE = "title: none | text: {text}"
QUERY_TEMPLATE = "task: search result | query: {question}"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = (
    torch.bfloat16
    if DEVICE == "cuda" and torch.cuda.is_bf16_supported()
    else torch.float32
)

PACKAGE_NAMES = ["torch", "numpy", "sentence-transformers", "transformers", "faiss-cpu", "huggingface-hub"]
PACKAGE_VERSIONS = {name: package_metadata.version(name) for name in PACKAGE_NAMES}
print("裝置：", DEVICE, "／模型精度：", DTYPE)
if DEVICE == "cuda":
    print("GPU：", torch.cuda.get_device_name(0))
display(pd.DataFrame(PACKAGE_VERSIONS.items(), columns=["套件", "版本"]))

CORE_MANIFEST_FILENAME = "core_manifest.json"
PENDING_MANIFEST_FILENAME = "pending_manifest.json"
CORE_EXPORT_ZIP = "rag-data.zip"
PENDING_EXPORT_ZIP = "rag-data-pending.zip"

## 3. 擷取與校驗函式（改編處）

- `extract_rag_text()`：老師版原函式不動——取標題＋【內容】正文；找不到非空正文即明確報錯，不擅自把整份文件拿去 embedding。pending 檔內的「背景參照」prelude 屬【內容】一部分，依交付設計原樣納入（已標明非本項本文），RAG 02 引用時以 chunk_id 為準。
- `parse_txt_header()`：讀取法規 txt 標頭欄位（law_name／article／paragraph／chapter／version_date／effective_status／effective_date／source／retrieved_at／chunk_id）。
- `load_package()`：老師版安全解壓（僅 TXT、擋隱藏檔與 __MACOSX、擋路徑穿越、擋重複檔名）＋本案 join 校驗：每檔必以 chunk_id 對上同包 manifest，否則 fail-closed；標頭與 manifest 不一致即報錯；manifest 有條目但未上傳者警告跳過（小樣本模式）。

In [ ]:
def extract_rag_text(raw_text):
    # 從 TXT 取出標題與正文；老師版原函式。
    normalized = raw_text.lstrip("\ufeff").replace("\r\n", "\n").replace("\r", "\n")
    title_match = re.search(r"^標題[：:]\s*(.+)$", normalized, flags=re.MULTILINE)
    content_match = re.search(
        r"^【內容】[ \t]*\n(?P<body>.*?)(?=^【[^】\n]+】|\Z)",
        normalized,
        flags=re.MULTILINE | re.DOTALL,
    )
    if content_match is None or not content_match.group("body").strip():
        raise ValueError("找不到非空的【內容】正文；請檢查語料格式。")
    title = title_match.group(1).strip() if title_match else ""
    body = content_match.group("body").strip()
    return f"{title}\n\n{body}" if title else body


def read_header_field(normalized, name):
    match = re.search(rf"^{re.escape(name)}[：:][ \t]*(.*)$", normalized, flags=re.MULTILINE)
    return match.group(1).strip() or None if match else None


def parse_txt_header(raw_text):
    normalized = raw_text.lstrip("\ufeff").replace("\r\n", "\n").replace("\r", "\n")
    header = {}
    for name in ["law_name", "article", "paragraph", "chapter", "version_date",
                 "effective_status", "effective_date", "source", "retrieved_at", "chunk_id"]:
        header[name] = read_header_field(normalized, name)
    header["title"] = read_header_field(normalized, "標題")
    return header


def normalize_paragraph(value):
    # 全文條：txt 寫 all，manifest 寫 null；兩者等價。
    if value in (None, "", "all"):
        return None
    return str(value)


def compose_article_label(article, paragraph):
    paragraph = normalize_paragraph(paragraph)
    if paragraph is None:
        return f"第{article}條"
    return f"第{article}條第{paragraph}項"


def effective_date_display(value):
    # 條件式顯示（SDD v1.0 修訂）：是真實日期即顯示該日期；否則為未定。
    if value and re.match(r"^\d{4}-\d{2}-\d{2}$", str(value)):
        return str(value)
    return "未定"


def validate_manifest_entry(entry, package):
    chunk_id = entry.get("chunk_id", "")
    if not chunk_id.startswith(package + ":"):
        raise ValueError(f"manifest 條目 {chunk_id} 不屬 {package} 包（前綴錯誤）。")
    expected_status = "effective" if package == "core" else "not_yet_effective"
    if entry.get("effective_status") != expected_status:
        raise ValueError(
            f"{chunk_id} 生效狀態 {entry.get('effective_status')} 與 {package} 包預期 {expected_status} 不符，拒絕混入。"
        )
    if not entry.get("article") or not entry.get("file"):
        raise ValueError(f"{chunk_id} 缺 article 或 file 欄位，拒絕索引。")


def safe_unzip_txt(archive_bytes):
    # 老師版安全解壓：僅 TXT，擋路徑穿越與重複檔名。
    data_dir = Path(tempfile.mkdtemp(prefix="rag_documents_"))
    with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
        members = [
            member for member in archive.infolist()
            if not member.is_dir()
            and member.filename.lower().endswith(".txt")
            and not any(part.startswith((".", "__MACOSX")) for part in Path(member.filename).parts)
        ]
        if not members:
            raise ValueError("ZIP 中找不到 TXT。")
        destinations = set()
        for member in members:
            target = (data_dir / member.filename).resolve()
            if not target.is_relative_to(data_dir.resolve()):
                raise ValueError(f"ZIP 包含不適當的檔案路徑：{member.filename}")
            if target in destinations:
                raise ValueError(f"ZIP 有重複檔名：{member.filename}")
            destinations.add(target)
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(member))
    txt_paths = sorted(
        [p for p in data_dir.rglob("*") if p.is_file() and p.suffix.lower() == ".txt"],
        key=lambda p: p.relative_to(data_dir).as_posix(),
    )
    return data_dir, txt_paths


def load_package(uploaded, package, manifest_filename):
    # 載入一包語料：zip＋manifest join 校驗，fail-closed。回傳 (documents, stats)。
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(zip_names) != 1:
        raise ValueError(f"{package} 包：請一次上傳一個語料 ZIP 檔（目前 {len(zip_names)} 個）。")
    if manifest_filename not in uploaded:
        raise ValueError(f"{package} 包：請同時上傳 {manifest_filename}（repo rag-data/ 下載）。")
    archive_name = zip_names[0]
    archive_bytes = uploaded[archive_name]
    archive_sha256 = hashlib.sha256(archive_bytes).hexdigest()
    manifest = json.loads(uploaded[manifest_filename])
    if manifest.get("package") != package:
        raise ValueError(f"{manifest_filename} 的 package 為 {manifest.get('package')}，與 {package} 包不符。")
    entries_by_chunk = {entry["chunk_id"]: entry for entry in manifest["chunks"]}
    data_dir, txt_paths = safe_unzip_txt(archive_bytes)
    documents = []
    warnings = []
    for path in txt_paths:
        try:
            raw = path.read_text(encoding="utf-8-sig")
        except UnicodeDecodeError as exc:
            raise ValueError(f"{path.name} 不是可讀取的 UTF-8 文字。") from exc
        header = parse_txt_header(raw)
        chunk_id = header.get("chunk_id")
        if not chunk_id:
            raise ValueError(f"{path.name} 標頭缺 chunk_id，拒絕索引（fail-closed）。")
        entry = entries_by_chunk.get(chunk_id)
        if entry is None:
            raise ValueError(f"{chunk_id} 在 {manifest_filename} 無對應條目，拒絕索引（fail-closed）。")
        validate_manifest_entry(entry, package)
        if entry.get("file") != path.name:
            raise ValueError(f"{chunk_id} manifest file 欄位 {entry.get('file')} 與實際檔名 {path.name} 不符。")
        for field, header_value, entry_value in [
            ("article", header.get("article"), str(entry.get("article"))),
            ("paragraph", normalize_paragraph(header.get("paragraph")), normalize_paragraph(entry.get("paragraph"))),
            ("chapter", header.get("chapter"), entry.get("chapter")),
            ("version_date", header.get("version_date"), entry.get("version_date")),
            ("effective_status", header.get("effective_status"), entry.get("effective_status")),
            ("effective_date", header.get("effective_date"), str(entry.get("effective_date"))),
        ]:
            if header_value is not None and str(header_value) != str(entry_value):
                raise ValueError(f"{chunk_id} 標頭 {field}（{header_value}）與 manifest（{entry_value}）不一致。")
        rag_text = extract_rag_text(raw)
        paragraph = normalize_paragraph(entry.get("paragraph"))
        documents.append({
            "doc_id": chunk_id,
            "text": rag_text,
            "metadata": {
                "chunk_id": chunk_id,
                "title": header.get("title") or entry.get("file"),
                "article_label": compose_article_label(entry.get("article"), paragraph),
                "article": str(entry.get("article")),
                "paragraph": paragraph,
                "chapter": entry.get("chapter"),
                "version_date": entry.get("version_date"),
                "effective_status": entry.get("effective_status"),
                "effective_date": entry.get("effective_date"),
                "effective_date_display": effective_date_display(entry.get("effective_date")),
                "source_url": entry.get("source_url"),
                "source_ref": None,
                "source_ref_status": "missing（manifest 無此欄，見驗收報告 FINDING-01，不捏造）",
                "context_refs": entry.get("context_refs") or [],
                "file": entry.get("file"),
                "law_name": entry.get("law_name"),
                "retrieved_at": entry.get("retrieved_at"),
            },
        })
    loaded_ids = {doc["doc_id"] for doc in documents}
    if len(loaded_ids) != len(documents):
        raise ValueError(f"{package} 包：doc_id 重複。")
    skipped = sorted(set(entries_by_chunk) - loaded_ids)
    if skipped:
        warnings.append(f"manifest {len(entries_by_chunk)} 條中 {len(skipped)} 條未上傳檔案（小樣本模式警告跳過）：{skipped}")
    stats = {
        "archive_name": archive_name,
        "archive_sha256": archive_sha256,
        "manifest_entries": len(entries_by_chunk),
        "files_loaded": len(documents),
        "skipped": skipped,
        "warnings": warnings,
    }
    return documents, stats

## 4. 上傳 core 語料包

一次上傳兩個檔案：core 語料 zip ＋ core_manifest.json（repo rag-data/ 下載）。manifest 為全量覆蓋表（147 條），小樣本僅上傳 5 檔時，未上傳條目會警告跳過，不報錯。

In [ ]:
from google.colab import files

uploaded_core = files.upload()
CORE_DOCUMENTS, CORE_STATS = load_package(uploaded_core, "core", CORE_MANIFEST_FILENAME)
for warning in CORE_STATS["warnings"]:
    print("警告：", warning)
print("core：manifest", CORE_STATS["manifest_entries"], "條／載入", CORE_STATS["files_loaded"], "檔")
print("來源 zip：", CORE_STATS["archive_name"])
display(pd.DataFrame([
    {"chunk_id": doc["doc_id"], "條號": doc["metadata"]["article_label"],
     "章": doc["metadata"]["chapter"], "生效狀態": doc["metadata"]["effective_status"],
     "字元數": len(doc["text"])}
    for doc in CORE_DOCUMENTS
]))
assert all(doc["metadata"]["effective_status"] == "effective" for doc in CORE_DOCUMENTS)
assert all(doc["doc_id"].startswith("core:") for doc in CORE_DOCUMENTS)

## 5. 上傳 pending 隔離包

與 core 完全分開的路徑與變數；pending 絕不寫入 core 索引。一次上傳兩個檔案：pending 語料 zip ＋ pending_manifest.json。

In [ ]:
uploaded_pending = files.upload()
PENDING_DOCUMENTS, PENDING_STATS = load_package(uploaded_pending, "pending", PENDING_MANIFEST_FILENAME)
for warning in PENDING_STATS["warnings"]:
    print("警告：", warning)
print("pending：manifest", PENDING_STATS["manifest_entries"], "條／載入", PENDING_STATS["files_loaded"], "檔")
print("來源 zip：", PENDING_STATS["archive_name"])
display(pd.DataFrame([
    {"chunk_id": doc["doc_id"], "條號": doc["metadata"]["article_label"],
     "章": doc["metadata"]["chapter"], "生效狀態": doc["metadata"]["effective_status"],
     "施行日期顯示": doc["metadata"]["effective_date_display"], "字元數": len(doc["text"])}
    for doc in PENDING_DOCUMENTS
]))
assert all(doc["metadata"]["effective_status"] == "not_yet_effective" for doc in PENDING_DOCUMENTS)
assert all(doc["doc_id"].startswith("pending:") for doc in PENDING_DOCUMENTS)

## 6. 載入 embedding 模型

- 模型與老師版相同：google/embeddinggemma-2，SentenceTransformer 載入。
- 若模型下載要求授權，請以 Colab Secrets 設定 HF_TOKEN 後重跑本格（勿將 token 寫入 notebook）。
- `MODEL_REVISION` 取不到時記為 None，不阻斷建庫（版本資訊以 config.json 的 package_versions 為準）。

In [ ]:
from transformers import AutoTokenizer

model = SentenceTransformer(MODEL_ID, device=DEVICE)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
try:
    MODEL_REVISION = HfApi().model_info(MODEL_ID).sha
except Exception as exc:
    print("取不到模型 revision，記為 None：", exc)
    MODEL_REVISION = None
DIMENSION = model.get_sentence_embedding_dimension()
print("向量維度：", DIMENSION)


def count_input_tokens(text):
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])


def embed_documents(documents):
    texts = [DOCUMENT_TEMPLATE.format(text=doc["text"]) for doc in documents]
    for doc, text in zip(documents, texts):
        n_tokens = count_input_tokens(text)
        if n_tokens > TOKEN_LIMIT:
            raise ValueError(f"{doc['doc_id']} 輸入 {n_tokens} tokens，超過上限 {TOKEN_LIMIT}。")
    vectors = model.encode(
        texts, prompt="", convert_to_numpy=True,
        normalize_embeddings=True, batch_size=BATCH_SIZE, show_progress_bar=False,
    )
    vectors = np.ascontiguousarray(vectors, dtype=np.float32)
    if vectors.shape != (len(documents), DIMENSION) or not np.isfinite(vectors).all():
        raise ValueError("文件向量的維度或數值不正確。")
    if (np.linalg.norm(vectors, axis=1) <= 0).any():
        raise ValueError("有文件產生零向量。")
    faiss.normalize_L2(vectors)
    return vectors


def embed_question(question):
    if not isinstance(question, str) or not question.strip():
        raise ValueError("請輸入非空白問題。")
    query_input = QUERY_TEMPLATE.format(question=question.strip())
    if count_input_tokens(query_input) > TOKEN_LIMIT:
        raise ValueError("問題超過模型輸入長度上限，請縮短問題。")
    query_vector = model.encode(
        [query_input], prompt="", convert_to_numpy=True,
        normalize_embeddings=True, show_progress_bar=False,
    )
    query_vector = np.ascontiguousarray(query_vector, dtype=np.float32)
    if query_vector.shape != (1, DIMENSION) or not np.isfinite(query_vector).all():
        raise ValueError("問題向量的維度或數值不正確。")
    if np.linalg.norm(query_vector) <= 0:
        raise ValueError("問題產生零向量。")
    faiss.normalize_L2(query_vector)
    return query_vector

## 7. 建立 core 主庫索引

老師版：IndexFlatIP 精確搜尋；雙方皆正規化，內積排名即 cosine similarity 排名。

In [ ]:
core_embeddings = embed_documents(CORE_DOCUMENTS)
core_index = faiss.IndexFlatIP(DIMENSION)
core_index.add(core_embeddings)
core_doc_ids = [doc["doc_id"] for doc in CORE_DOCUMENTS]
core_by_id = {doc["doc_id"]: doc for doc in CORE_DOCUMENTS}
assert core_index.ntotal == len(core_doc_ids) == len(CORE_DOCUMENTS)
print("core 索引類型：", type(core_index).__name__)
print("core 索引向量數：", core_index.ntotal)
print("core 索引維度：", core_index.d)

## 8. 建立 pending 隔離庫索引

與 core 完全獨立的索引物件；兩庫不共用索引、不共用 doc_id 命名空間（前綴區隔）。

In [ ]:
pending_embeddings = embed_documents(PENDING_DOCUMENTS)
pending_index = faiss.IndexFlatIP(DIMENSION)
pending_index.add(pending_embeddings)
pending_doc_ids = [doc["doc_id"] for doc in PENDING_DOCUMENTS]
pending_by_id = {doc["doc_id"]: doc for doc in PENDING_DOCUMENTS}
assert pending_index.ntotal == len(pending_doc_ids) == len(PENDING_DOCUMENTS)
print("pending 索引類型：", type(pending_index).__name__)
print("pending 索引向量數：", pending_index.ntotal)
print("pending 索引維度：", pending_index.d)

## 9. 檢索函式與試查

- 搜尋函式只做檢索，不呼叫 LLM。精確搜尋保證依目前向量找到分數最高者，不保證文件一定足以回答問題（老師版提醒）。
- 試查三項皆為檢索層驗證：T1 數字題取回、T4 連動、pending 隔離。T2 查無題不得幻覺由 RAG 02 生成層負責，此處僅確認 core 庫取不到 pending 條文。

In [ ]:
def search_index(index, by_id, ids, question, top_k=3):
    if isinstance(top_k, bool) or not isinstance(top_k, (int, np.integer)) or top_k < 1:
        raise ValueError("top_k 必須是大於 0 的整數。")
    query_vector = embed_question(question)
    scores, positions = index.search(query_vector, min(int(top_k), index.ntotal))
    results = []
    for rank, (position, score) in enumerate(zip(positions[0], scores[0]), start=1):
        doc_id = ids[int(position)]
        results.append({"rank": rank, "score": float(score), **by_id[doc_id]})
    return results


# 試查 1（T1 數字題）：拒絕酒測的罰鍰，預期取回 core:art35_para4（十八萬元）。
trial_1 = search_index(core_index, core_by_id, core_doc_ids, "汽車駕駛人拒絕酒測，罰鍰多少？", top_k=3)
display(pd.DataFrame([
    {"排名": r["rank"], "chunk_id": r["doc_id"], "條號": r["metadata"]["article_label"],
     "cosine_similarity": round(r["score"], 6)}
    for r in trial_1
]))
assert trial_1[0]["doc_id"] == "core:art35_para4", f"試查 1 首位非預期：{trial_1[0]['doc_id']}"

# 試查 2（T4 連動）：art35_para4 的 context_refs 須指回同包 art35_para1，且該檔在小樣本內。
para4 = core_by_id["core:art35_para4"]
print("core:art35_para4 context_refs：", para4["metadata"]["context_refs"])
assert "core:art35_para1" in para4["metadata"]["context_refs"]
assert "core:art35_para1" in core_by_id
print("T4 連動目標 core:art35_para1 在小樣本內：", core_by_id["core:art35_para1"]["metadata"]["article_label"])

# 試查 3（pending 隔離）：以 pending 專屬概念問 core 庫，取回不得含 pending 條文；問 pending 庫則須取回。
trial_3_core = search_index(core_index, core_by_id, core_doc_ids, "汽車駕駛人體格變化應繳回駕照重新審查", top_k=3)
assert all(not r["doc_id"].startswith("pending:") for r in trial_3_core)
assert all(r["doc_id"] != "pending:art92-2" for r in trial_3_core)
trial_3_pending = search_index(pending_index, pending_by_id, pending_doc_ids, "汽車駕駛人體格變化應繳回駕照重新審查", top_k=3)
print("pending 庫試查首位：", trial_3_pending[0]["doc_id"], trial_3_pending[0]["metadata"]["article_label"])
assert trial_3_pending[0]["doc_id"] == "pending:art92-2"
print("試查通過：T1 取回、T4 連動、pending 隔離皆符合。")

## 10. 匯出與重載驗證（core）

老師版四檔格式不動：index.faiss ＋ documents.json ＋ index_doc_ids.json ＋ config.json；僅檔名改為作業七規定的 rag-data.zip，並在 config 增記 package 與來源 zip 資訊。重載驗證沿用老師版：雜湊、數量、ID 對照、保存前後檢索一致、FAISS 與 NumPy 全矩陣內積比對。

In [ ]:
def build_export(index, documents, ids, embeddings, package, stats, export_zip_name, verification_question):
    export_dir = Path(tempfile.mkdtemp(prefix="rag_index_"))
    faiss.write_index(index, str(export_dir / "index.faiss"))
    for filename, data in [("documents.json", documents), ("index_doc_ids.json", ids)]:
        (export_dir / filename).write_text(
            json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8"
        )
    config = {
        "format_version": "1.0",
        "package": package,
        "created_at": datetime.now(ZoneInfo("Asia/Taipei")).isoformat(timespec="seconds"),
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "model_config_kwargs": {"vision_config": None, "audio_config": None},
        "dimension": DIMENSION,
        "normalize_embeddings": True,
        "index_type": "IndexFlatIP",
        "metric": "cosine_similarity_via_normalized_inner_product",
        "document_template": DOCUMENT_TEMPLATE,
        "query_template": QUERY_TEMPLATE,
        "encode_prompt": "",
        "max_seq_length": TOKEN_LIMIT,
        "document_count": len(documents),
        "document_unit": "one_document_one_vector",
        "document_text_includes_title": True,
        "package_versions": PACKAGE_VERSIONS,
        "build_device": DEVICE,
        "build_dtype": str(DTYPE),
        "source_archive": stats["archive_name"],
        "source_archive_sha256": stats["archive_sha256"],
        "source_manifest_entries": stats["manifest_entries"],
        "source_files_loaded": stats["files_loaded"],
        "file_sha256": {
            name: hashlib.sha256((export_dir / name).read_bytes()).hexdigest()
            for name in ["index.faiss", "documents.json", "index_doc_ids.json"]
        },
    }
    (export_dir / "config.json").write_text(
        json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(f"已保存（{package}）：")
    for path in sorted(export_dir.iterdir()):
        print(path.name, f"({path.stat().st_size:,} bytes)")

    # 重載驗證（老師版）：雜湊、數量、ID 對照、保存前後一致、精確搜尋比對。
    loaded_config = json.loads((export_dir / "config.json").read_text(encoding="utf-8"))
    for name, expected_hash in loaded_config["file_sha256"].items():
        actual_hash = hashlib.sha256((export_dir / name).read_bytes()).hexdigest()
        assert actual_hash == expected_hash, f"檔案完整性檢查失敗：{name}"
    loaded_index = faiss.read_index(str(export_dir / "index.faiss"))
    loaded_documents = json.loads((export_dir / "documents.json").read_text(encoding="utf-8"))
    loaded_ids = json.loads((export_dir / "index_doc_ids.json").read_text(encoding="utf-8"))
    assert loaded_index.ntotal == len(loaded_documents) == len(loaded_ids) == loaded_config["document_count"]
    assert loaded_index.d == loaded_config["dimension"]
    assert len(set(loaded_ids)) == len(loaded_ids)
    assert set(loaded_ids) == {doc["doc_id"] for doc in loaded_documents}
    assert loaded_documents == documents and loaded_ids == ids
    verification_query = embed_question(verification_question)
    k = min(3, index.ntotal)
    before_scores, before_positions = index.search(verification_query, k)
    after_scores, after_positions = loaded_index.search(verification_query, k)
    np.testing.assert_array_equal(before_positions, after_positions)
    np.testing.assert_allclose(before_scores, after_scores, atol=1e-6, rtol=1e-6)
    all_scores = (embeddings @ verification_query[0]).ravel()
    expected_top_scores = np.sort(all_scores)[::-1][:k]
    np.testing.assert_allclose(after_scores[0], expected_top_scores, atol=1e-5, rtol=1e-5)
    np.testing.assert_allclose(after_scores[0], all_scores[after_positions[0]], atol=1e-5, rtol=1e-5)
    print("通過：檔案完整性、文件對照、索引維度、保存前後一致，以及精確搜尋檢查。")

    export_zip = export_dir.parent / f"{export_dir.name}_download" / export_zip_name
    export_zip.parent.mkdir(parents=True, exist_ok=True)
    required_files = ["index.faiss", "documents.json", "index_doc_ids.json", "config.json"]
    with zipfile.ZipFile(export_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for name in required_files:
            archive.write(export_dir / name, arcname=name)
    with zipfile.ZipFile(export_zip) as archive:
        assert archive.testzip() is None
        assert set(archive.namelist()) == set(required_files)
    print("下載檔名：", export_zip_name)
    print("大小：", f"{export_zip.stat().st_size:,} bytes")
    files.download(str(export_zip))
    return export_zip


EXPORT_ZIP_CORE = build_export(
    core_index, CORE_DOCUMENTS, core_doc_ids, core_embeddings,
    "core", CORE_STATS, CORE_EXPORT_ZIP, "汽車駕駛人拒絕酒測，罰鍰多少？",
)

## 11. 匯出與重載驗證（pending）

與 core 完全相同的四檔格式與驗證流程，僅 package 與檔名不同（rag-data-pending.zip）。兩包在任何步驟皆不共用索引物件與 doc_id。

In [ ]:
EXPORT_ZIP_PENDING = build_export(
    pending_index, PENDING_DOCUMENTS, pending_doc_ids, pending_embeddings,
    "pending", PENDING_STATS, PENDING_EXPORT_ZIP, "汽車駕駛人體格變化應繳回駕照重新審查",
)

## 12. 交接清單（PM 後續）

- 下載上述兩個 zip，上傳雲端轉成可下載網址，網址填入 RAG 02 的下載 cell（repo 為 private，勿用 GitHub raw link）。
- 本次執行版（帶 output）由 PM 存入 colab/executed/（檔名帶日期），為繳交註三留痕證據；repo 主檔維持清 output。
- source_ref 溯源缺口（manifest 無此欄、raw/ 尚無 PDF）見 tests/rag01_acceptance_report.md FINDING-01，待 HO／Instinct 處理，不影響本次建庫。

In [ ]:
for label, path in [("core", EXPORT_ZIP_CORE), ("pending", EXPORT_ZIP_PENDING)]:
    data = path.read_bytes()
    print(f"{label}：{path.name}／{len(data):,} bytes／sha256 {hashlib.sha256(data).hexdigest()}")